In [0]:
%sql

create database retail_sales_analytics.sales

In [0]:
%sql

create volume retail_sales_analytics.sales.Files

In [0]:
raw_orders = spark.read.csv("/Volumes/retail_sales_analytics/sales/files/Bronze/retail_dataset.csv",header = "True" , inferSchema = "True")
display(raw_orders)

order_id,order_date,customer_id,customer_name,product_id,product_name,category,quantity,price,payment_type,order_status,returned
1001,2025-07-28,C004,Customer_4,P103,Denim Jean,Apparel,1,339.97,card,shipped,no
1002,2025-09-09,C054,Customer_54,P117,USB Cable,Accessories,1,$380.61,cash,completed,no
1003,2025-08-13,C059,Customer_59,P101,T-shirt XL,Apparel,3,278.26,card,completed,no
1004,11-Sep-2025,C013,Customer_13,P107,Charger,Accessories,3,$85.51,upi,completed,no
1005,24/08/2025,C035,Customer_35,P112,Backpack,Accessories,1,343.88,cash,cancelled,no
1006,09-16-2025,C028,Customer_28,P110,Watch,Accessories,3,$75.72,upi,cancelled,no
1007,27/07/2025,C064,Customer_64,P116,keyboard,electronics,2,315.59,upi,cancelled,no
1008,2025-08-10,C069,Customer_69,P103,Denim Jean,Apparel,null,$150.28,upi,delivered,no
1009,06/09/2025,C065,Customer_65,P109,Jeans,Apparel,-1,271.99,cash,cancelled,yes
1010,2025-09-18,C031,Customer_31,P109,jeans,Apparel,1,$47.4,card,cancelled,no


In [0]:
raw_customers = spark.read.json("/Volumes/retail_sales_analytics/sales/files/Bronze/customer_dataset.json")
display(raw_customers)


age,city,customer_id,gender,loyalty_tier,signup_date
35.0,Delhi,C001,M,Gold,01-Apr-2020
39.0,Gurgaon,C002,null,silver,14/11/2023
null,Kolkata,C003,F,Silver,2022/09/13
null,BENGALURU,C004,Male,null,2017-02-21
19.0,Pune,C005,Male,PLATINUM,2022/06/06
37.0,Hyderabad,C006,f,null,27-Apr-2018
50.0,Kolkata,C007,M,null,2021/11/02
59.0,Bangalore,C008,null,Silver,17-Jun-2021
22.0,BENGALURU,C009,F,null,06-May-2020
58.0,Hyderabad,C010,M,Gold,12/12/2019


In [0]:
from pyspark.sql.functions import *
spark.conf.set("spark.sql.ansi.enabled", "false")

order_ts = coalesce(
    to_timestamp(trim(col("order_date")), "yyyy-MM-dd"),
    to_timestamp(trim(col("order_date")), "dd/MM/yyyy"),
    to_timestamp(trim(col("order_date")), "d-MMM-yyyy"),
    to_timestamp(trim(col("order_date")), "MM-dd-yyyy"),
    to_timestamp(trim(col("order_date")), "dd-MM-yyyy")
)

clean_orders = (
    raw_orders
    .withColumn("order_id", trim(col("order_id")).cast("long"))
    .withColumn("order_date_ts", order_ts)
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("product_name", lower(trim(col("product_name"))))
    .withColumn("category", lower(trim(col("category"))))
    .withColumn("quantity", when(trim(col("quantity")).isNull() | (trim(col("quantity"))==""), lit(1)).otherwise(trim(col("quantity"))))
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("quantity", when(col("quantity") <= 0, lit(1)).otherwise(col("quantity")))
    .withColumn("price_clean", translate(trim(col("price")), "$,", ""))
    .withColumn("price", col("price_clean").cast("double"))
    .drop("price_clean")
    .withColumn("payment_type", lower(trim(col("payment_type"))))
    .withColumn("order_status", lower(trim(col("order_status"))))
    .withColumn("returned", lower(trim(col("returned"))))
    .withColumn("total_amount", (coalesce(col("quantity"), lit(0)) * coalesce(col("price"), lit(0.0))).cast("double"))
)

clean_orders = clean_orders.dropDuplicates(["order_id", "product_id"]).filter(col("order_id").isNotNull() & col("product_id").isNotNull())

# display(clean_orders)

clean_orders.write.format("delta").mode("overwrite").saveAsTable("retail_sales_analytics.sales.silver_orders")


In [0]:
# ---------- SILVER: Customers ----------

signup_ts = coalesce(
    to_timestamp(trim(col("signup_date")), "yyyy-MM-dd"),
    to_timestamp(trim(col("signup_date")), "dd/MM/yyyy"),
    to_timestamp(trim(col("signup_date")), "d-MMM-yyyy"),
    to_timestamp(trim(col("signup_date")), "yyyy/MM/dd")
)

clean_customers = (
    raw_customers
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("gender", lower(trim(col("gender"))))
    .withColumn("age", when(trim(col("age")).isNull() | (trim(col("age"))==""), None).otherwise(col("age")))
    .withColumn("age", when(col("age").cast('int') < 0, None).otherwise(col("age").cast('int')))
    .withColumn("city", trim(col("city")))
    .withColumn("loyalty_tier", lower(trim(col("loyalty_tier"))))
    .withColumn("signup_date", signup_ts)
)

# display(clean_customers)

clean_customers.write.format("delta").mode("overwrite").saveAsTable("retail_sales_analytics.sales.silver_customers")



In [0]:
from pyspark.sql.functions import sum as _sum, min as _min, max as _max, avg, countDistinct, coalesce, when, lit, current_timestamp, to_date
silver_orders = spark.table("retail_sales_analytics.sales.silver_orders")
silver_customers = spark.table("retail_sales_analytics.sales.silver_customers")

orders_enriched = silver_orders.join(silver_customers, on="customer_id", how="left")

orders_enriched = orders_enriched.withColumn("order_date", to_date(col("order_date_ts"))) \
                                 .withColumn("year", year(col("order_date_ts"))) \
                                 .withColumn("month", month(col("order_date_ts")))

gold_df = (
    orders_enriched
    .groupBy("product_id","product_name","customer_id","category","year","month","order_date","gender","age","city","loyalty_tier")
    .agg(
        _sum("total_amount").alias("total_sales"),
        _sum(coalesce(col("quantity"), lit(0))).alias("total_quantity"),
        countDistinct("order_id").alias("total_orders"),
        _min("price").alias("min_price"),
        _max("price").alias("max_price"),
        avg("price").alias("avg_unit_price"),
        _sum(when(col("returned") == "yes", 1).otherwise(0)).alias("returned_count"),
        _sum(when(col("returned") == "yes", col("total_amount")).otherwise(0.0)).alias("returned_amount")
    )
)

gold_df = gold_df.withColumn("avg_order_value", when(col("total_orders")>0, col("total_sales")/col("total_orders")).otherwise(lit(0.0))) \
                 .withColumn("avg_price_per_item", when(col("total_quantity")>0, col("total_sales")/col("total_quantity")).otherwise(lit(0.0))) \
                 .withColumn("refreshed_at", current_timestamp())

# display(gold_df)

gold_df.write.format("delta").mode("overwrite").saveAsTable("retail_sales_analytics.sales.gold_aggregates")


In [0]:
%sql

select * from retail_sales_analytics.sales.gold_aggregates

product_id,product_name,customer_id,category,year,month,order_date,gender,age,city,loyalty_tier,total_sales,total_quantity,total_orders,min_price,max_price,avg_unit_price,returned_count,returned_amount,avg_order_value,avg_price_per_item,refreshed_at
P101,t-shirt xl,C049,apparel,2025,8,2025-08-23,male,26,Delhi,null,298.99,1,1,298.99,298.99,298.99,0,0.0,298.99,298.99,2026-08-22T20:30:56.952Z
P101,t-shirt xl,C032,apparel,2025,8,2025-08-12,null,null,Pune,silver,140.62,1,1,140.62,140.62,140.62,0,0.0,140.62,140.62,2026-08-22T20:30:56.952Z
P110,watch,C031,accessories,2025,8,2025-08-31,m,34,Lucknow,platinum,629.13,3,1,209.71,209.71,209.71,0,0.0,629.13,209.71,2026-08-22T20:30:56.952Z
P111,socks,C011,apparel,2025,9,2025-09-17,m,21,delhi,gold,308.91,1,1,308.91,308.91,308.91,0,0.0,308.91,308.91,2026-08-22T20:30:56.952Z
P102,sneakers,C035,shoes,2025,8,2025-08-27,female,39,Mumbai,platinum,210.02,1,1,210.02,210.02,210.02,1,210.02,210.02,210.02,2026-08-22T20:30:56.952Z
P107,charger,C012,accessories,2025,8,2025-08-28,female,52,Noida,platinum,474.29999999999995,3,1,158.1,158.1,158.1,0,0.0,474.29999999999995,158.1,2026-08-22T20:30:56.952Z
P117,usb cable,C054,accessories,2025,9,2025-09-09,f,26,delhi,gold,380.61,1,1,380.61,380.61,380.61,0,0.0,380.61,380.61,2026-08-22T20:30:56.952Z
P117,usb cable,C036,accessories,2025,8,2025-08-23,m,48,Gurgaon,gold,448.4,1,1,448.4,448.4,448.4,0,0.0,448.4,448.4,2026-08-22T20:30:56.952Z
P109,jeans,C025,apparel,2025,8,2025-08-22,m,23,mumbai,silver,309.52,1,1,309.52,309.52,309.52,0,0.0,309.52,309.52,2026-08-22T20:30:56.952Z
P115,mouse,C030,electronics,2025,7,2025-07-22,null,39,null,bronze,261.35,1,1,261.35,261.35,261.35,0,0.0,261.35,261.35,2026-08-22T20:30:56.952Z
